# TP4: Chatbots basados en recuperación de la información

## Punto 1
1.1)
### Definicion de la aplicacion
Aplicacion: asistente virtual de hardware para principiantes. El chatbot esta diseñado para ayudar a niños, adultos mayores o personas sin conocimientos tecnicos a entender las partes de una computadora y que necesitan para armar una.


1.2)
## Listado de preguntas y respuestas

| #  | Pregunta                                      | Respuesta                                                                 |
|----|-----------------------------------------------|---------------------------------------------------------------------------|
| 1  | ¿Qué es el cerebro de la computadora?         | El cerebro es el procesador o CPU. Es el que da todas las órdenes para que la máquina funcione. |
| 2  | ¿Para qué sirve la memoria RAM?               | La RAM sirve para que la computadora pueda hacer muchas cosas al mismo tiempo sin trabarse. |
| 3  | ¿Dónde se guardan mis fotos y videos?         | Se guardan en el disco rígido o en un SSD, que es como un cajón donde guardas tus cosas para siempre. |
| 4  | ¿Qué es una tarjeta de video o placa de video?| Es la parte que se encarga de que los dibujos y los juegos se vean bien en la pantalla. |
| 5  | ¿Cómo se conectan todas las piezas?           | Todas las piezas se conectan en la "Placa Madre", que es como el piso de la computadora. |
| 6  | ¿Por qué la computadora necesita una fuente?  | La fuente de poder le da la electricidad necesaria para que todas las partes puedan encenderse. |
| 7  | ¿Qué hace que la computadora no se caliente?  | Se usan ventiladores o "coolers" que tiran aire frío para que las piezas no se quemen. |
| 8  | ¿Qué es un gabinete?                          | Es la caja de metal o plástico donde se meten todas las piezas para que estén protegidas. |
| 9  | ¿Necesito un monitor para usar la PC?         | Sí, el monitor es la pantalla donde ves todo lo que la computadora está haciendo. |
| 10 | ¿Cómo escribo y muevo la flechita?            | Usas el teclado para escribir y el ratón o mouse para mover la flechita en la pantalla. |
| 11 | ¿Qué es un SSD?                               | Es un tipo de disco muy rápido que hace que la computadora prenda en poquitos segundos. |
| 12 | ¿Puedo armar una computadora yo solo?         | Sí, con cuidado y siguiendo un manual, es como armar un juego de bloques de construcción. |
| 13 | ¿Qué pasa si le cae agua adentro?             | ¡Cuidado! El agua puede quemar las piezas eléctricas. Siempre mantené los líquidos lejos. |
| 14 | ¿Para qué sirven los cables de colores?       | Son los que llevan la corriente desde la fuente a cada parte de la computadora. |
| 15 | ¿Qué es Intel o AMD?                          | Son las marcas que fabrican los procesadores (los cerebros de la computadora). |
| 16 | ¿Cómo escucho música en la PC?                | Necesitas conectar parlantes o auriculares en los agujeritos de sonido de la computadora. |
| 17 | ¿Qué es el sistema operativo?                 | Es el programa principal, como Windows o Linux, que te permite usar la computadora fácilmente. |
| 18 | ¿Por qué hace ruido mi computadora?           | Suele ser el ruido de los ventiladores girando para sacar el calor hacia afuera. |
| 19 | ¿Qué es un procesador con gráficos integrados?| Es un cerebro que ya viene con la capacidad de mostrar dibujos sin necesitar una placa de video aparte. |
| 20 | ¿Dónde enchufo el cable de internet?          | Se enchufa en un huequito especial llamado puerto de red o Ethernet que está atrás de la caja. |


In [ ]:
!pip install spacy --quiet
!python -m download es_core_news_sm --quiet
!python -m spacy download es_core_news_sm --quiet

In [ ]:
!pip install nltk

In [ ]:
# Celda 1: Instalación de dependencias en el entorno de Google Colab
!pip install langchain langchain-community langchain-huggingface faiss-cpu dnspython

In [ ]:
import es_core_news_sm
nlp = es_core_news_sm.load()

# Punto 2
### Crear el chatbot utilizando TF-IDF y similitud del coseno.

In [ ]:
# @title
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

# Cargamos las palabras vacias em español para que el bot las ignore
palabras_vacias_es = stopwords.words('spanish')

# Definimos el dataset
preguntas = [
    "¿Qué es el cerebro de la computadora?",
    "¿Para qué sirve la memoria RAM?",
    "¿Dónde se guardan mis fotos y videos?",
    "¿Qué es una tarjeta de video o placa de video?",
    "¿Cómo se conectan todas las piezas?",
    "¿Por qué la computadora necesita una fuente?",
    "¿Qué hace que la computadora no se caliente?",
    "¿Qué es un gabinete?",
    "¿Necesito un monitor para usar la PC?",
    "¿Cómo escribo y muevo la flechita?",
    "¿Qué es un SSD?",
    "¿Puedo armar una computadora yo solo?",
    "¿Qué pasa si le cae agua adentro?",
    "¿Para qué sirven los cables de colores?",
    "¿Qué es Intel o AMD?",
    "¿Cómo escucho música en la PC?",
    "¿Qué es el sistema operativo?",
    "¿Por qué hace ruido mi computadora?",
    "¿Qué es un procesador con gráficos integrados?",
    "¿Dónde enchufo el cable de internet?"
]
respuestas = [
    "El cerebro es el procesador o CPU. Es el que da todas las órdenes para que la máquina funcione.",
    "La RAM sirve para que la computadora pueda hacer muchas cosas al mismo tiempo sin trabarse.",
    "Se guardan en el disco rígido o en un SSD, que es como un cajón donde guardas tus cosas para siempre.",
    "Es la parte que se encarga de que los dibujos y los juegos se vean bien en la pantalla.",
    "Todas las piezas se conectan en la 'Placa Madre', que es como el piso de la computadora.",
    "La fuente de poder le da la electricidad necesaria para que todas las partes puedan encenderse.",
    "Se usan ventiladores o 'coolers' que tiran aire frío para que las piezas no se quemen.",
    "Es la caja de metal o plástico donde se meten todas las piezas para que estén protegidas.",
    "Sí, el monitor es la pantalla donde ves todo lo que la computadora está haciendo.",
    "Usas el teclado para escribir y el ratón o mouse para mover la flechita en la pantalla.",
    "Es un tipo de disco muy rápido que hace que la computadora prenda en poquitos segundos.",
    "Sí, con cuidado y siguiendo un manual, es como armar un juego de bloques de construcción.",
    "¡Cuidado! El agua puede quemar las piezas eléctricas. Siempre mantené los líquidos lejos.",
    "Son los que llevan la corriente desde la fuente a cada parte de la computadora.",
    "Son las marcas que fabrican los procesadores (los cerebros de la computadora).",
    "Necesitas conectar parlantes o auriculares en los agujeritos de sonido de la computadora.",
    "Es el programa principal, como Windows o Linux, que te permite usar la computadora fácilmente.",
    "Suele ser el ruido de los ventiladores girando para sacar el calor hacia afuera.",
    "Es un cerebro que ya viene con la capacidad de mostrar dibujos sin necesitar una placa de video aparte.",
    "Se enchufa en un huequito especial llamado puerto de red o Ethernet que está atrás de la caja."
]

# Creamos el vectorizador TF-IDF y transformamos nuestras preguntas
vectorizador = TfidfVectorizer(stop_words=palabras_vacias_es)
tfidf_matrix = vectorizador.fit_transform(preguntas)

# Funcion para procesar la consulta del usuario y encontrar la mejor respuesta
def chatbot_tfidf(consulta_usuario):
  # Transformamons la pregunta del usuario al mismo formato vectorial
  consulta_vec = vectorizador.transform([consulta_usuario])

  # Calculamos la similitud del coseno entre la consulta y todas las preguntas
  similitudes = cosine_similarity(consulta_vec, tfidf_matrix).flatten()

  # Buscamos el indice de la pregunta con mayor similitud
  indice_mejor_coincidencia = similitudes.argmax()

  # Si la similitud es muy baja (ejemplo 0), decimos que no entendismos
  if similitudes[indice_mejor_coincidencia] == 0:
    return "Lo siento, no entiendo tu pregunta de hardware"

  return respuestas[indice_mejor_coincidencia]
'''
# Ejemplo de uso
pregunta_usuario = input("Hazme una consulta sobre computacion o hardware")
print(f"Usuario: {pregunta_usuario}")
print(f"Chatbot: {chatbot_tfidf(pregunta_usuario)}")
'''

## Aclaraciones y conclusiones
Se hizo una prueba del modelo y se detecto que fallaba bastante.
Se concluyo que el fallo se debio a que el modelo se baso en palabras muy repetidas en el dataset, como "para", "la" y "de".

Si no se filtran palabras vacias (stop word) el modelo termina eligiendo la respuesta que matematicamente comparte mas conectores gramaticales, ignorando el verdadero significado del la pregunta.

Se procedio a modificar el codigo sacando las "stop word"

# Punto 3
### Crear otro chatbot utilizando embeddings. Indique cuál embedding

- Seleccionamos el modelo pre entrenado es_core:news_sm de SpaCy.
- Cuando hacemos nlp(p).vector, la libreria toma el embedding (el vector numerico) de cada palabra de la frase y calcula un vector promedio que representa el significado global de la pregunta.
- Al igual que en el punto anterior, medimos la distancia angular entre los vactores utilizando cosine_similarity. La diferencia crucial es que aqui estamos comparando significados y no solo caracteres o palabras identicas


In [ ]:
!python -m spacy download es_core_news_md

In [ ]:
import spacy
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Cargamos el modelo en español de Spacy que descargamos al inicio del notebook
# Este modelo contiene los vectores (embeddings) para el vocabulario en español.
nlp = spacy.load("es_core_news_md")

# Reutilizamos el mismo dataset de 20 preguntas y respuestas del Punto 1
preguntas = [
    "¿Qué es el cerebro de la computadora?",
    "¿Para qué sirve la memoria RAM?",
    "¿Dónde se guardan mis fotos y videos?",
    "¿Qué es una tarjeta de video o placa de video?",
    "¿Cómo se conectan todas las piezas?",
    "¿Por qué la computadora necesita una fuente?",
    "¿Qué hace que la computadora no se caliente?",
    "¿Qué es un gabinete?",
    "¿Necesito un monitor para usar la PC?",
    "¿Cómo escribo y muevo la flechita?",
    "¿Qué es un SSD?",
    "¿Puedo armar una computadora yo solo?",
    "¿Qué pasa si le cae agua adentro?",
    "¿Para qué sirven los cables de colores?",
    "¿Qué son los procesadores Intel o AMD?",
    "¿Cómo escucho música en la PC?",
    "¿Qué es el sistema operativo?",
    "¿Por qué hace ruido mi computadora?",
    "¿Qué es un procesador con gráficos integrados?",
    "¿Dónde enchufo el cable de internet?"
]

respuestas = [
    "El cerebro es el procesador o CPU. Es el que da todas las órdenes para que la máquina funcione.",
    "La RAM sirve para que la computadora pueda hacer muchas cosas al mismo tiempo sin trabarse.",
    "Se guardan en el disco rígido o en un SSD, que es como un cajón donde guardas tus cosas para siempre.",
    "Es la parte que se encarga de que los dibujos y los juegos se vean bien en la pantalla.",
    "Todas las piezas se conectan en la 'Placa Madre', que es como el piso de la computadora.",
    "La fuente de poder le da la electricidad necesaria para que todas las partes puedan encenderse.",
    "Se usan ventiladores o 'coolers' que tiran aire frío para que las piezas no se quemen.",
    "Es la caja de metal o plástico donde se meten todas las piezas para que estén protegidas.",
    "Sí, el monitor es la pantalla donde ves todo lo que la computadora está haciendo.",
    "Usas el teclado para escribir y el ratón o mouse para mover la flechita en la pantalla.",
    "Es un tipo de disco muy rápido que hace que la computadora prenda en poquitos segundos.",
    "Sí, con cuidado y siguiendo un manual, es como armar un juego de bloques de construcción.",
    "¡Cuidado! El agua puede quemar las piezas eléctricas. Siempre mantené los líquidos lejos.",
    "Son los que llevan la corriente desde la fuente a cada parte de la computadora.",
    "Son las marcas que fabrican los procesadores (los cerebros de la computadora).",
    "Necesitas conectar parlantes o auriculares en los agujeritos de sonido de la computadora.",
    "Es el programa principal, como Windows o Linux, que te permite usar la computadora fácilmente.",
    "Suele ser el ruido de los ventiladores girando para sacar el calor hacia afuera.",
    "Es un cerebro que ya viene con la capacidad de mostrar dibujos sin necesitar una placa de video aparte.",
    "Se enchufa en un huequito especial llamado puerto de red o Ethernet que está atrás de la caja."
]

# Pre calculamos los embedding promedio para cada una de nuestras 20 preguntas
# Spacy  calcula automaticamente el vector promedio de una frase promediando los vectores de sus palabras
vectores_preguntas = [nlp(p).vector for p in preguntas]

# Funcion del chatbot basado en embeddings y similitudes del coseno
def chatbot_embeddings(consulta_usuario):
  # Obtenemos el vector embedding de la consulta del usuario
  vector_usuario = nlp(consulta_usuario).vector.reshape(1, -1)

  # Armamos una lista vacia
  similitudes = []
  # Calculamos la similitud del coseno de la consulta contra cada pregunta guardada
  for vec_p in vectores_preguntas:
    vec_p_resf = vec_p.reshape(1, -1)
    sim = cosine_similarity(vector_usuario, vec_p_resf)[0][0]
    similitudes.append(sim)

  # Buscamos el indice con la mayor similitud semantica
  indice_mejor_coincidencia = np.argmax(similitudes)

  # Umbral minimo de confianza (para evitar respuestas aleatorias)
  if similitudes[indice_mejor_coincidencia] < 0.4:
    return "Lo siento, no encontre nada parecido en mis conocimientos"
  return respuestas[indice_mejor_coincidencia]

'''
pregunta_usuario = input("Hazme una consulta sobre computacion o hardware")
print(f"Usuario: {pregunta_usuario}")
print(f"Chatbot: {chatbot_embeddings(pregunta_usuario)}")
'''


## Aclaraciones y conclusiones
Al igual que en el modelo anterior se hizo una prueba y tambien fallaba bastante.

Se concluyo que la falla se debia a que cargamos el modelo es_core_news_sm que es un modelo muy chico (sm significa small)

Este modelo no incluye vectores de palabras reales (embedding estaticos) de alta calidad, solo trae pequeños tensores optimizados para tareas basicas.

Al hacer nlp(consultas).vector, el modelo genera un vector pobre o aproximado por contexto gramatical.

Se soluciono cambiando el modelo


---

# Punto 5
Demostracion de ambos cahtbot con prueba estandar de 3 preguntas

In [ ]:
# 1. Definimos las preguntas de prueba que usara un usuario principiante
consultas_prueba = [
    "¿Para que sirve la memoria RAM?",
    "Necesito una pantalla para ver los dibujos",
    "¿Que hacer si el cerebro intel o amd se caliente?"
]

print("=== EVALUACIÓN Y COMPARACIÓN DE CHATBOTS ===\n")

for i, consulta in enumerate(consultas_prueba, 1):
    print(f"Prueba {i}: '{consulta}'")
    print("-" * 50)

    # Probamos el Chatbot de TF-IDF
    respuesta_tfidf = chatbot_tfidf(consulta)
    print(f"[TF-IDF]   -> Bot responde: {respuesta_tfidf}")

    # Probamos el Chatbot de Embeddings
    respuesta_emb = chatbot_embeddings(consulta)
    print(f"[EMBEDDING]-> Bot responde: {respuesta_emb}")
    print("\n" + "="*60 + "\n")

# Conclusiones

## TF-IDF
El modelo TF-IDF es un buscador de coincidencias, su exito en las pruebas 1 y 2 demuestran que si se usan las palabras exactas, el modelo la va a encontrar de forma matematica muy rapida, pero es propenso a confundirse si acumulas muchas palabras claves juntas, como lo demuestra la prueba 3, donde "Intel" y "AMD" tapo la necesidad de caliente

## Embeddings
Los embeddings promediados miran el contexto global, esto quedo demostrado en la prueba 3, donde el chatbot entendio perfectamente que el termino "se caliente" requeria la respuesta de los coolers

En la prueba 1 al ser muy similar la pregunta realizada con la del vector pregunta, el chatbot entendio perfecto que respuesta dar.

En la prueba 2, el chatbot fallo, esto se debe que al promediar todas las palabras de la oracion, las palabras vacias o las estructuras de los verbos pueden desviar el vector hacia respuestas con estructuras gramaticales similares.

---

# TP5: Chatbots basados en Generacion Aumentada por Recuperacion (RAG)

### a) Creación del conjunto de datos de evaluación.
Ademas del dataset original que creo, ahora crearemos un dataset de prueba o evaluacion con la misma logica: preguntas y respuestas.


In [ ]:
# Dataset de Evaluacion (Punto 1.a)
# Mantiene el mismo formato de listas separadas del TP4, pero con frases alternativas
preguntas_evaluacion = [
    "¿Que marca de procesador me recomendás para la compu?",
    "Mi maquina emite mucho calor, ¿qué le tengo que comprar?",
    "Quiero guardar los videos de mi cumpleaños para siempre, ¿donde van?",
    "¿Necesito comprar una televisión o algo parecido para poder usarla?",
    "¿Cómo hago para que la flechita de la pantalla se mueva?",
    "¿Es muy difícil armarla uno mismo en casa?",
    "¿Qué pasa si mi gato tira la taza de té sobre la caja?",
    "¿Para qué sirven todos esos cables que salen de la caja de electricidad?",
    "¿Qué es eso de Windows o Linux que me dicen que instale?",
    "¿Puedo usar internet si no tengo el cablecito negro enchufado atrás?"
]

respuestas_esperadas_evaluacion = [
    "Las marcas mas conocidas del mercado son AMD e INTEL",
    "Se usan ventiladores o 'coolers' que tiran aire frío para que las piezas no se quemen.",
    "Se guardan en el disco rígido o en un SSD, que es como un cajón donde guardas tus cosas para siempre.",
    "Sí, el monitor es la pantalla donde ves todo lo que la computadora está haciendo.",
    "Usas el teclado para escribir y el ratón o mouse para mover la flechita en la pantalla.",
    "Sí, con cuidado y siguiendo un manual, es como armar un juego de bloques de construcción.",
    "¡Cuidado! El agua puede quemar las piezas eléctricas. Siempre mantené los líquidos lejos.",
    "Son los que llevan la corriente desde la fuente a cada parte de la computadora.",
    "Es el programa principal, como Windows o Linux, que te permite usar la computadora fácilmente.",
    "Se puede usar una placa USB wifi. Se enchufa en un puerto USB y permite conectarse a internet usando redes WIFI, sin cable"
]

print(f"Dataset de evaluación inicializado. Preguntas: {len(preguntas_evaluacion)} | Respuestas: {len(respuestas_esperadas_evaluacion)}")

### b) Seleccion y Justificacion de Modelos

#### 1. Modelo LLM Seleccionado: `Qwen/Qwen2.5-1.5B-Instruct`
* **Justificación:** Se optó por un Modelo de Lenguaje Pequeño (SLM) de última generación, especializado en seguir instrucciones (*Instruct*) y con un excelente soporte nativo para el idioma español. Su arquitectura compacta permite realizar inferencias rápidas y eficientes al cargarse directamente en el entorno local de Google Colab. Actuará como nuestro motor de generación de respuestas (Generación en RAG), tomando el contexto recuperado y adaptándolo al lenguaje amigable requerido por los usuarios principiantes.

#### 2. Modelo de Embedding 1 (Especializado/Multilingüe): `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`
* **Justificación:** Este modelo fue entrenado específicamente para mapear oraciones de múltiples idiomas (incluyendo el español) en un espacio vectorial denso común. Captura con gran precisión la similitud semántica y conceptual subyacente. Es el candidato ideal para nuestro asistente de hardware, ya que debería asociar términos informales (ej. *"cerebro"*) con los términos técnicos correctos (ej. *"procesador"*).

#### 3. Modelo de Embedding 2 (Base Multilingüe General): `distilbert-base-multilingual-cased`
* **Justificación:** Se selecciona este modelo basado en una arquitectura BERT destilada y balanceada para múltiples idiomas. A diferencia del Modelo 1, este es un codificador de propósito general que no cuenta con un ajuste fino específico para la tarea de similitud de oraciones o paráfrasis. Su inclusión nos permitirá evaluar experimentalmente si un modelo especializado en estructuras de diálogo (Modelo 1) ofrece un mejor rendimiento en la recuperación (*Retrieval*) frente a un modelo lingüístico general (Modelo 2) trabajando sobre el mismo idioma.

### c) Implemente una clase ChatBot usando lo elegido en b).
Podemos usar cualquier base de datos vectorial: Chroma y FAISS son las más documentadas.

In [ ]:
from langchain_core.documents import Document

# 1. Traemos la base de conocimiento original del TP4
preguntas_tp4 = [
    "¿Qué es el cerebro de la computadora?", "¿Para qué sirve la memoria RAM?", "¿Dónde se guardan mis fotos y videos?",
    "¿Qué es una tarjeta de video o placa de video?", "¿Cómo se conectan todas las piezas?", "¿Por qué la computadora necesita una fuente?",
    "¿Qué hace que la computadora no se caliente?", "¿Qué es un gabinete?", "¿Necesito un monitor para usar la PC?",
    "¿Cómo escribo y muevo la flechita?", "¿Qué es un SSD?", "¿Puedo armar una computadora yo solo?",
    "¿Qué pasa si le cae agua adentro?", "¿Para qué sirven los cables de colores?", "¿Qué es Intel o AMD?",
    "¿Cómo escucho música en la PC?", "¿Qué es el sistema operativo?", "¿Por qué hace ruido mi computadora?",
    "¿Qué es un procesador con gráficos integrados?", "¿Dónde enchufo el cable de internet?"
]

respuestas_tp4 = [
    "El cerebro es el procesador o CPU. Es el que da todas las órdenes para que la máquina funcione.",
    "La RAM sirve para que la computadora pueda hacer muchas cosas al mismo tiempo sin trabarse.",
    "Se guardan en el disco rígido o en un SSD, que es como un cajón donde guardas tus cosas para siempre.",
    "Es la parte que se encarga de que los dibujos y los juegos se vean bien en la pantalla.",
    "Todas las piezas se conectan en la 'Placa Madre', que es como el piso de la computadora.",
    "La fuente de poder le da la electricidad necesaria para que todas las partes puedan encenderse.",
    "Se usan ventiladores o 'coolers' que tiran aire frío para que las piezas no se quemen.",
    "Es la caja de metal o plástico donde se meten todas las piezas para que estén protegidas.",
    "Sí, el monitor es la pantalla donde ves todo lo que la computadora está haciendo.",
    "Usas el teclado para escribir y el ratón o mouse para mover la flechita en la pantalla.",
    "Es un tipo de disco muy rápido que hace que la computadora prenda en poquitos segundos.",
    "Sí, con cuidado y siguiendo un manual, es como armar un juego de bloques de construcción.",
    "¡Cuidado! El agua puede quemar las piezas eléctricas. Siempre mantené los líquidos lejos.",
    "Son los que llevan la corriente desde la fuente a cada parte de la computadora.",
    "Son las marcas que fabrican los procesadores (los cerebros de la computadora).",
    "Necesitas conectar parlantes o auriculares en los agujeritos de sonido de la computadora.",
    "Es el programa principal, como Windows o Linux, que te permite usar la computadora fácilmente.",
    "Suele ser el ruido de los ventiladores girando para sacar el calor hacia afuera.",
    "Es un cerebro que ya viene con la capacidad de mostrar dibujos sin necesitar una placa de video aparte.",
    "Se enchufa en un huequito especial llamado puerto de red o Ethernet que está atrás de la caja."
]

# 2. Convertimos nuestro conocimiento del TP4 en una lista de objetos "Document" de LangChain.
# Esto es obligatorio porque las bases de datos vectoriales no aceptan texto plano (strings comunes).
documentos_conocimiento = []

for preg, resp in zip(preguntas_tp4, respuestas_tp4):
    # Ingeniería de Prompts: Juntamos la pregunta y la respuesta en un solo bloque de texto.
    # De esta forma, el buscador vectorial tendrá tanto la forma en que la gente pregunta
    # como la explicación técnica, todo junto en un mismo lugar para no perder contexto.
    texto_combinado = f"Duda común: {preg}\nExplicación: {resp}"

    # Creamos el objeto Document de LangChain. Tiene dos partes:
    # - page_content: El texto real que la IA va a procesar y leer.
    # - metadata: Una etiqueta o "ficha de control" interna (en este caso indicando la materia).
    doc = Document(page_content=texto_combinado, metadata={"materia": "Procesamiento del Habla"})

    # Guardamos este documento estructurado dentro de nuestra lista general.
    documentos_conocimiento.append(doc)

# Verificación visual para estar seguros de que se procesaron los 20 elementos.
print(f"Se procesaron {len(documentos_conocimiento)} documentos de conocimiento para la base vectorial.")

inicializar los dos modelos de embeddings de Hugging Face que elegimos y justificamos en el punto 1.b.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Inicializamos el primer modelo de Embedding (Especializado en Paráfrasis)
# Este modelo es el que se encarga de entender intenciones y relacionar sinónimos en español.
print("Descargando e inicializando Embedding 1 (paraphrase-multilingual)...")

embedding_parasis = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 2. Inicializamos el segundo modelo de Embedding (Base Multilingüe General)
# Este modelo entiende español pero no fue entrenado específicamente para buscar preguntas y respuestas similares.
print("Descargando e inicializando Embedding 2 (distilbert-multilingual)...")

embedding_general = HuggingFaceEmbeddings(
    model_name="distilbert-base-multilingual-cased"
)

print("\n¡Ambos modelos de embeddings han sido cargados con éxito en tu entorno venv_issd!")

### Crear las Bases de Datos Vectoriales
Ahora que tenemos los textos estructurados (documentos_conocimiento) y los modelos matemáticos de embeddings listos (embedding_parasis y embedding_general), es momento de unirlos para crear dos bases de datos vectoriales separadas usando la librería FAISS.

FAISS (Facebook AI Similarity Search) se va a encargar de tomar cada documento, pasarlo por el embedding correspondiente, convertirlo en una lista de números (vectores) y organizar el índice para permitir búsquedas semánticas ultrarrápidas.

In [ ]:
from langchain_community.vectorstores import FAISS

# 1. Creamos la Base de Datos Vectorial 1 (Usando el Embedding especializado en paráfrasis)
print("Indexando documentos en la Base Vectorial 1 (Paráfrasis)...")
db_parasis = FAISS.from_documents(documentos_conocimiento, embedding_parasis)

# 2. Creamos la Base de Datos Vectorial 2 (Usando el Embedding multilingüe general)
print("Indexando documentos en la Base Vectorial 2 (General)...")
db_general = FAISS.from_documents(documentos_conocimiento, embedding_general)

print("\n¡Ambas bases de datos vectoriales FAISS han sido creadas y guardadas en memoria!")

### Inicializar el Modelo de Lenguaje Local (Qwen) mediante Transformers

## Definir la función del ChatBot RAG

Esta función va a hacer tres cosas simples cada vez que le hagamos una pregunta:

- **Buscar:** Va a ir a la base de datos vectorial que le pasemos y va a traer el fragmento de texto más relevante ($k=1$).

- **Armar el Prompt:** Va a inyectar ese fragmento junto con la pregunta en una plantilla del sistema (system prompt), dándole el rol educativo y amigable.

- **Generar:** Va a procesar el prompt a través de nuestro modelo local cargado en la notebook y nos va a devolver la respuesta limpia.

In [ ]:
import warnings
warnings.filterwarnings("ignore") # Silencia las advertencias molestas de tokens en la pantalla
!pip install transformers accelerate --quiet

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
import torch

print("Descargando e inicializando Qwen2.5-1.5B desde Hugging Face...")

# 1. Definimos el ID del modelo oficial en Hugging Face
id_modelo = "Qwen/Qwen2.5-1.5B-Instruct"

# 2. Descargamos el tokenizador y el modelo usando la configuración óptima para Colab
tokenizer = AutoTokenizer.from_pretrained(id_modelo)
model = AutoModelForCausalLM.from_pretrained(
    id_modelo,
    torch_dtype="auto",
    device_map="auto" # Colab asigna automáticamente si usa CPU o GPU
)

# 3. Armamos el pipeline nativo de Hugging Face
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=120,
    temperature=0.2,
    do_sample=True,
    clean_up_tokenization_spaces=False
)

# 4. Lo convertimos en un componente compatible con LangChain
llm_huggingface_local = HuggingFacePipeline(pipeline=pipe)

print("\n✅ ¡Modelo de Hugging Face cargado con éxito en la memoria de Colab!")

In [ ]:
def ejecutar_rag_chatbot(pregunta_usuario, base_vectorial, componente_llm):
    # 1. PASO RETRIEVAL: Buscamos en la base de datos vectorial FAISS
    resultados = base_vectorial.similarity_search(pregunta_usuario, k=1)
    contexto_recuperado = resultados[0].page_content if resultados else "Sin contexto relevante."

    # 2. PASO PROMPT: Estructuramos el prompt educativo
    prompt = (
        "<|im_start|>system\n"
        "Sos un asistente virtual educativo de computación y hardware para principiantes (niños y abuelos).\n"
        "Tu objetivo es responder la duda de forma muy clara, cortita, dulce y amigable.\n"
        "REGLA ESTRICTA: Basate ÚNICAMENTE en el Contexto de Referencia brindado.\n\n"
        f"Contexto de Referencia:\n{contexto_recuperado}<|im_end|>\n"
        f"<|im_start|>user\n{pregunta_usuario}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

    # 3. PASO GENERACIÓN: Invocamos al modelo directamente
    respuesta = componente_llm.invoke(prompt)

    # Limpiamos el texto generado para quedarnos solo con la respuesta del asistente
    if "<|im_start|>assistant\n" in respuesta:
        respuesta = respuesta.split("<|im_start|>assistant\n")[-1]

    return respuesta.strip()

print("🤖 Función del ChatBot RAG vinculada al modelo de Hugging Face con éxito.")

In [ ]:
# Recorremos el dataset de evaluación para comparar el desempeño de ambos embeddings
print("=*" * 40)
print("             EVALUACIÓN COMPARATIVA DE EMBEDDINGS (RAG SYSTEM)             ")
print("=*" * 40 + "\n")

# Usamos enumerate para recorrer las preguntas que definiste en el punto 1.a
for indice, pregunta_test in enumerate(preguntas_evaluacion):
    respuesta_docente = respuestas_esperadas_evaluacion[indice]

    print(f"📌 [PREGUNTA {indice + 1}] — Evaluación de Usuario:")
    print(f"👉 '{pregunta_test}'")
    print(f"🎯 Respuesta Esperada (Ideal): {respuesta_docente}")
    print("-" * 50)

    # 1. Probamos con la Base Vectorial 1 (Especializada en Paráfrasis)
    print("🤖 MODELO 1 (paraphrase-multilingual) responde:")
    respuesta_m1 = ejecutar_rag_chatbot(pregunta_test, db_parasis, llm_huggingface_local)
    print(f"   ↳ {respuesta_m1}")
    print("." * 50)

    # 2. Probamos con la Base Vectorial 2 (Base Multilingüe General)
    print("🤖 MODELO 2 (distilbert-multilingual) responde:")
    respuesta_m2 = ejecutar_rag_chatbot(pregunta_test, db_general, llm_huggingface_local)
    print(f"   ↳ {respuesta_m2}")

    print("\n" + "="*80 + "\n")

print("📊 ¡Evaluación finalizada para todas las muestras del dataset!")

## Análisis Crítico de los Resultados

Si observamos las respuestas, hay un claro ganador en cuanto a precisión de contexto (Retrieval) y consistencia semántica: el Modelo 1 (paraphrase-multilingual).


## Conclusión y Comparativa de Desempeño
Ganador de la Evaluación: Modelo 1 (paraphrase-multilingual-MiniLM-L12-v2)

1. Justificación Técnica del Modelo 1 (Paráfrasis)
El modelo especializado en paráfrasis demostró una capacidad notable para capturar la intención semántica del usuario, abstrayéndose de la literalidad de las palabras.

Ejemplo Clave (Pregunta 10): Ante la consulta informal ¿Puedo usar internet si no tengo el cablecito negro enchufado atrás?, el Modelo 1 recuperó correctamente el contexto de redes inalámbricas y respondió hablando específicamente de Wi-Fi.

Ejemplo Clave (Pregunta 8): Identificó con precisión que "los cables que salen de la caja de electricidad" hacían referencia a la fuente de alimentación, desglosando componentes como el procesador y la RAM.

2. Comportamiento del Modelo 2 (Distilbert General)
El modelo distilbert-base-multilingual-cased es un modelo de lenguaje general. Al no estar optimizado para tareas de búsqueda de pasajes o mapeo de preguntas/respuestas, tiende a fallar cuando el usuario se expresa con lenguaje coloquial o metáforas.

Falla de Contexto (Pregunta 4): Cuando el usuario preguntó si necesitaba una televisión para usar la compu, el Modelo 2 recuperó un contexto erróneo y generó una alucinación cómica: "La computadora tiene su propio monitor llamado pantalla de teclado y mouse". Mezcló los periféricos al no entender la equivalencia conceptual entre TV y Monitor.

Falla de Contexto (Pregunta 10): Al responder sobre el cable de red, el Modelo 2 se desvió por completo sugiriendo usar el teléfono móvil y revisar la batería, perdiendo el foco del contexto de hardware de escritorio.

## Conclusiones Generales del Trabajo Práctico

Tras el desarrollo, implementación y evaluación experimental de nuestro sistema de Recuperación Aumentada por Generación (RAG), se extraen las siguientes conclusiones clave:

### 1. Impacto de la Especialización del Modelo de Embedding
El experimento demostró de forma contundente la importancia de seleccionar un modelo de embedding adecuado para la tarea específica:
*   **Modelo 1 (`paraphrase-multilingual-MiniLM-L12-v2`):** Mostró un desempeño óptimo al resolver de manera exitosa la **similitud semántica**. Logró asociar correctamente expresiones coloquiales e informales de los usuarios principiantes (ej. *"cerebro"*, *"dibujos"*) con los términos técnicos almacenados en la base de conocimiento (ej. *"procesador"*, *"tarjeta de video"*).
*   **Modelo 2 (`distilbert-base-multilingual-cased`):** Al ser un codificador lingüístico general no ajustado para paráfrasis, dependió excesivamente de la coincidencia exacta de palabras clave. Esto provocó fallas en el *Retrieval* ante preguntas con terminología informal, entregando contextos erróneos o insuficientes al LLM.

### 2. Viabilidad y Robustez del Despliegue Local (SLM)
La transición forzada desde una arquitectura basada en API hacia una ejecución **100% local** mediante la librería `transformers` y la asignación dinámica de hardware de Google Colab resultó ser un acierto crítico para el proyecto:
*   **Mitigación de Errores Externos:** Se eliminaron por completo las fallas de conexión por red (*timeouts*), las restricciones de cuotas de uso y la inestabilidad de los endpoints gratuitos de Hugging Face.
*   **Eficiencia de Qwen2.5-1.5B-Instruct:** Al tratarse de un Modelo de Lenguaje Pequeño (SLM) de última generación, demostró que una arquitectura compacta es más que suficiente para comprender las instrucciones del sistema (*System Prompt*) y procesar el contexto de referencia sin degradar el tiempo de inferencia, manteniendo las respuestas en un tono dulce, amigable y estrictamente acotado a la base de conocimiento provista.

En resumen, la combinación de un embedding especializado en estructuras de diálogo multilingüe junto con un SLM local de instrucciones constituye una arquitectura eficiente, predecible y de alto rendimiento para el desarrollo de asistentes virtuales educativos.